# Depth Model Comparison for Corn Seed Images

Benchmark 6 monocular depth estimation models on 200 random corn seed images.
Evaluates: edge alignment, seed/background contrast, intra-seed smoothness, and speed.

**GPU:** A100 recommended. T4 (free) also works.

**Models:** Depth Anything V2, Depth Pro (Apple), Pixel-Perfect Depth, VGGT (Meta), Depth Anything V3, Marigold LCM

## 1. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b colab https://github.com/shurjo05/DepthComparison.git /content/DepthComparison 2>/dev/null || echo "Already cloned"
%cd /content/DepthComparison

## 2. Set Dataset Path

**Edit the path below** to point to your corn seed dataset on Google Drive.
The folder should contain `images/` and `labels/` subdirectories.

In [ ]:
import os

# >>> EDIT THIS PATH to match your Google Drive folder <<<
os.environ["DATASET_DIR"] = "/content/drive/MyDrive/CornSeedDetection6D/test/test"

# Verify the dataset exists
img_dir = os.path.join(os.environ["DATASET_DIR"], "images")
lbl_dir = os.path.join(os.environ["DATASET_DIR"], "labels")
assert os.path.isdir(img_dir), f"Images dir not found: {img_dir}"
assert os.path.isdir(lbl_dir), f"Labels dir not found: {lbl_dir}"
n_images = len([f for f in os.listdir(img_dir) if f.endswith(".jpg")])
print(f"Found {n_images} images in {img_dir}")

## 3. Install Dependencies & Clone Model Repos

In [ ]:
!pip install -q -r requirements.txt

# Clone model repos that need separate installation
!git clone https://github.com/gangweix/pixel-perfect-depth 2>/dev/null || echo "pixel-perfect-depth already cloned"
!git clone https://github.com/facebookresearch/vggt 2>/dev/null || echo "vggt already cloned"
!git clone https://github.com/ByteDance-Seed/Depth-Anything-3 2>/dev/null || echo "Depth-Anything-3 already cloned"

# Install repo-specific deps
!cd pixel-perfect-depth && pip install -q -r requirements.txt
!cd vggt && pip install -q -r requirements.txt
!cd Depth-Anything-3 && pip install -q -e .

!nvidia-smi

## 4. Select 200 Random Sample Images

In [ ]:
!python depth_comparison/select_samples.py

## 5. Run All 6 Depth Models

Runs sequentially, clears GPU between models. ~30-60 min on T4, faster on A100.
To run specific models only: `--models depth_anything_v2 depth_pro marigold`

In [ ]:
!python depth_comparison/run_all_models.py

## 6. Evaluate & Generate Comparison Grids

In [ ]:
!python depth_comparison/evaluate.py

## 7. View Results

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image as IPImage
import glob, os

# Load and display metrics table
with open("depth_comparison/results_summary.json") as f:
    results = json.load(f)

df = pd.DataFrame(results).T
display(df)

# Display comparison grids
grids = sorted(glob.glob("depth_comparison/comparison_grids/*.png"))[:10]
for g in grids:
    print(f"\n{os.path.basename(g)}")
    display(IPImage(filename=g))